In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
import pickle
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

In [97]:
with open("../data/03-result/merged_df.pkl","rb") as f:
    merged_df =pickle.load(f)

In [98]:
# Consider dropping name embedding similarity since it contains SPOILERS
merged_df = merged_df.drop(columns = ["name_similarity"])

# Fit models

In [99]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# ---------------------------------------
# 1. Split predictors / label
# ---------------------------------------
X = merged_df.drop(columns=["label"])
y = merged_df["label"]

# Identify column types
num_cols = X.select_dtypes(include=["float64", "int64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# ---------------------------------------
# 2. Preprocessing pipelines
# ---------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# ---------------------------------------
# 3. Models
# ---------------------------------------
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestClassifier(class_weight='balanced',n_estimators=300, random_state=42))
])

logreg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("logreg", LogisticRegression(class_weight='balanced', max_iter=500))
])

# ---------------------------------------
# 4. Train/test split
# ---------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------
# 5. Fit models
# ---------------------------------------
rf_model.fit(X_train, y_train)
logreg_model.fit(X_train, y_train)

# ---------------------------------------
# 6. Predictions
# ---------------------------------------
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

logreg_pred = logreg_model.predict(X_test)
logreg_proba = logreg_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# ---------------------------------------
# 7. Evaluation
# ---------------------------------------

# TODO: balanced_accuracy score
# TODO: see wikipedian evalution metrics forbinary classification
# TODO: check imbalanced_learn module 

print("Random Forest:")
print("Confusion matrix:\n", confusion_matrix(y_test, rf_pred))
print("  Accuracy:", accuracy_score(y_test, rf_pred))
print("  Precision:", precision_score(y_test, rf_pred))
print("  Recall:", recall_score(y_test, rf_pred))
print("  F1 score:", f1_score(y_test, rf_pred))
print("  AUC:", roc_auc_score(y_test, rf_proba))

print("\nLogistic Regression:")
print("Confusion matrix:\n", confusion_matrix(y_test, logreg_pred))
print("  Accuracy:", accuracy_score(y_test, logreg_pred))
print("  Precision:", precision_score(y_test, logreg_pred))
print("  Recall:", recall_score(y_test, logreg_pred))
print("  F1 score:", f1_score(y_test, logreg_pred))
print("  AUC:", roc_auc_score(y_test, logreg_proba))

Random Forest:
Confusion matrix:
 [[  2  12]
 [  0 141]]
  Accuracy: 0.9225806451612903
  Precision: 0.9215686274509803
  Recall: 1.0
  F1 score: 0.9591836734693877
  AUC: 0.800405268490375

Logistic Regression:
Confusion matrix:
 [[  6   8]
 [ 15 126]]
  Accuracy: 0.8516129032258064
  Precision: 0.9402985074626866
  Recall: 0.8936170212765957
  F1 score: 0.9163636363636364
  AUC: 0.6033434650455927


In [100]:
import numpy as np

# Extract trained RF model inside the pipeline
rf = rf_model.named_steps["rf"]

# Get the transformed column names
num_features = list(num_cols)
cat_features = list(cat_cols)

# After transformation, numeric features stay single.
# Categorical features remain single because we didn't one-hot encode them.
all_features = num_features + cat_features

# Extract importances
importances = rf.feature_importances_

# Combine into one dataframe
rf_feature_importance = pd.DataFrame({
    "feature": all_features,
    "importance": importances
}).sort_values(by="importance", ascending=False)
print(rf_feature_importance) 

             feature  importance
169   disease_path_6    0.017365
202  disease_path_39    0.013867
192  disease_path_29    0.013822
183  disease_path_20    0.013498
130            FP_68    0.013399
..               ...         ...
2     inorganic_flag    0.000175
6         veterinary    0.000113
1     chemical_probe    0.000089
4             orphan    0.000023
0     biotherapeutic    0.000000

[213 rows x 2 columns]


In [101]:
logreg = logreg_model.named_steps["logreg"]

coef = logreg.coef_[0]   # Binary classification → one vector

logreg_feature_importance = pd.DataFrame({
    "feature": all_features,
    "coef": coef,
    "abs_coef": np.abs(coef)
}).sort_values(by="abs_coef", ascending=False)

print(logreg_feature_importance)

             feature      coef  abs_coef
171   disease_path_8  1.721484  1.721484
211  disease_path_48 -1.400206  1.400206
179  disease_path_16 -1.359374  1.359374
192  disease_path_29  1.271727  1.271727
191  disease_path_28 -1.202315  1.202315
..               ...       ...       ...
67              FP_5 -0.006312  0.006312
119            FP_57 -0.004237  0.004237
158            FP_96  0.001283  0.001283
173  disease_path_10 -0.000781  0.000781
0     biotherapeutic  0.000000  0.000000

[213 rows x 3 columns]


# TODO: PU / semi-supervized models 

# TODO: Graph models

In [102]:
# TODO: explore GNN, node2vec ....
from torch_geometric.data import HeteroData

data = HeteroData()

data['drug'].x = drug_features_tensor
data['disease'].x = disease_features_tensor
data['pathway'].x = pathway_dummy_features # can be zeros

# Edges
data['drug', 'interacts_with', 'pathway'].edge_index = drug_to_pathway_edges
data['disease', 'associated_with', 'pathway'].edge_index = disease_to_pathway_edges

# Now define a heterogeneous GNN
model = HeteroGNN(...)

# Training objective: predict edges between drug and disease
# You give positive pairs (known therapeutic links) and negative pairs (random)

NameError: name 'drug_features_tensor' is not defined

In [ ]:
final_diseases_df["phenotypes"][0]["rows"]

[{'phenotypeHPO': {'id': 'HP_0001370',
   'name': 'Rheumatoid arthritis',
   'description': 'Inflammatory changes in the synovial membranes and articular structures with widespread fibrinoid degeneration of the collagen fibers in mesenchymal tissues, as well as atrophy and rarefaction of bony structures.'}},
 {'phenotypeHPO': {'id': 'HP_0001945',
   'name': 'Fever',
   'description': 'Body temperature elevated above the normal range.'}},
 {'phenotypeHPO': {'id': 'HP_0003565',
   'name': 'Elevated erythrocyte sedimentation rate',
   'description': 'An increased erythrocyte sedimentation rate (ESR). The ESR is a test that measures the distance that erythrocytes have fallen after one hour in a vertical column of anticoagulated blood under the influence of gravity. The ESR is a nonspecific finding. An elevation may indicate inflammation or may be caused by any condition that elevates fibrinogen.'}},
 {'phenotypeHPO': {'id': 'HP_0011227',
   'name': 'Elevated circulating C-reactive protein 